# 04 · Find a cyclone feature in TopK, KAN, or KAN + TopK

**Run after notebook 03 and your chosen training notebook. Use the same Drive account and a T4 or larger GPU.** This branch defaults to `kan_topk_runs/kan_topk_k32/best.pt` from notebook 02c. Choose `SAE_TYPE='kan_sae'` for the original KAN run or `topk` for the original TopK baseline.

Fit one probe per frozen feature on 2019 storms, choose the feature and cutoff on 2020 storms, then score the locked choice on 2021 storms. Both KAN models preserve negative values. KAN + TopK applies its trained magnitude gate before probing; it does not probe the dense pre-gate spline values.

Use the same cohort and evaluation protocol across runs. Tune model settings on validation results; do not select them by repeatedly comparing test F1. Existing results on the original branch remain available; this notebook's outputs are cleared for the new experiment.


## 1. Connect Drive and load the checkpoint

In [1]:
# @title Connect Drive and load the project
import json
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

# Notebooks 02c, 04 and 05 share this immutable code version.
RELEASE = 'ddbc28bfbd8e74e4791e15253d3a504b1c745d4b'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/AMS691_GraphCast')
LABEL_RUN = DRIVE_PROJECT / 'cyclones_2019_2021'
DATA = REPO / 'data' / 'cyclone_2019_2021'


Mounted at /content/drive


In [2]:
# @title Select the saved model and run
COHORT = 'course'  # @param ["course", "all_guarded"]
SAE_TYPE = 'kan_topk'  # @param ["kan_topk", "kan_sae", "topk"]
SAE_RUN = ''  # @param {type:"string"}
SAE_EXPERIMENT = 'monthly_2017_2018'  # @param {type:"string"}
# A blank SAE_RUN selects the baseline name for the chosen model type.
RUNS = {'topk': ('topk_runs', 'topk_baseline'),
        'kan_sae': ('kan_runs', 'kan_baseline'),
        'kan_topk': ('kan_topk_runs', 'kan_topk_k32')}
RUN_GROUP, default_run = RUNS[SAE_TYPE]
SAE_RUN = SAE_RUN.strip() or default_run
for name in [SAE_RUN, SAE_EXPERIMENT]:
    if not name or Path(name).name != name or name in {'.', '..'}:
        raise ValueError('Use simple experiment and run folder names.')
checkpoint = DRIVE_PROJECT / SAE_EXPERIMENT / RUN_GROUP / SAE_RUN / 'best.pt'
from cyclone_probes import open_experiment, encode_split, fit_probes, select_feature
from cyclone_probes import save_selection, evaluate_selected, write_table, release_features
from cyclone_data import write_json
from IPython.display import display
import pandas as pd
import gc

model, device, manifest, y_by_time, identity = open_experiment(LABEL_RUN, DATA, COHORT, checkpoint)
if identity.get('model_type', 'topk') != SAE_TYPE:
    raise ValueError('The checkpoint model type does not match SAE_TYPE.')
OUT = LABEL_RUN / COHORT / 'probe_runs' / (SAE_RUN + '_' + identity['checkpoint_sha256'][:12])
LOCAL_FEATURE_CACHE = Path('/content/ams691_probe_features')
print(f"Loaded {SAE_TYPE} best.pt from epoch {identity['checkpoint_epoch']}. Device: {device}.")
print('Dictionary size:', identity['latent'])
if SAE_TYPE in {'topk', 'kan_topk'}:
    print('Maximum active features per row:', identity['k'])
print('Results folder:', OUT)


Loaded kan_topk best.pt from epoch 20. Device: cuda.
Dictionary size: 4096
Maximum active features per row: 32
Results folder: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/kan_topk_k32_13ce05f84137


## 2. Fit the probes on 2019 storms

Each feature gets a slope and intercept in `p = sigmoid(slope × activation + intercept)`. Cyclone and background rows have equal total weight in the training loss. Scale by the feature's **training maximum absolute value**, preserving its sign; apply the same small L2 penalty as before. The saved slope and cutoff use the original feature units.

Both TopK models use sparse storage, preserving signed KAN + TopK values. Original L1 KAN features can be dense, so exact float32 values are stored on Colab's local disk, one column at a time for fitting. With the default 4,096 features, the course fitting split needs about 8.1 GB of temporary disk. That cache is released when this cell finishes. It is not copied to Drive.

The fitter checks each solution and retries with a second optimizer when necessary. A numerical warning alone does not stop the run, but a probe that still fails the convergence checks raises an error. If updating this notebook after an optimizer error, restart the Colab session and run this notebook from the beginning; your saved SAE checkpoint and labeled activations are reused.

In [3]:
# @title Fit one logistic classifier per frozen SAE feature
X_fit, y_fit, _ = encode_split('fit', LABEL_RUN, manifest, y_by_time, model, device,
                              cache_dir=LOCAL_FEATURE_CACHE)
try:
    print(f'Fitting on {len(y_fit):,} mesh rows, including {y_fit.sum():,} cyclone rows.')
    weights = fit_probes(X_fit, y_fit)
finally:
    release_features(X_fit)
del X_fit, y_fit
gc.collect()


Encoding features:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting on 491,544 mesh rows, including 742 cyclone rows.


Fitting single-feature probes:   0%|          | 0/4096 [00:00<?, ?it/s]

417

## 3. Choose the feature and cutoff on 2020 storms

Compare all distinct signed activation values, correctly including zero where it occurs. A positive probe slope uses an inclusive `>=` rule; a negative slope uses `<=`. Validation F1 pools all nodes, including quiet maps. Ties within a feature prefer fewer predicted positives; feature ties prefer fewer false positives and then the smaller feature index.

For the original L1 KAN only, the course validation cache needs about 5.4 GB of temporary disk and is released after selection. Test labels do not participate in fitting or selection.

In [4]:
# @title Select and save the best feature using validation F1
X_val, y_val, _ = encode_split('validation', LABEL_RUN, manifest, y_by_time, model, device,
                              cache_dir=LOCAL_FEATURE_CACHE)
try:
    ranking = select_feature(X_val, y_val, weights)
    save_selection(OUT, identity, weights, ranking)
finally:
    release_features(X_val)
winner = ranking[0]
display(pd.DataFrame(ranking[:5])[['feature','rule','activation_cutoff','precision','recall','f1']])
print(f"Selected feature {winner['feature']}: predict cyclone when activation "
      f"{winner['rule']} {winner['activation_cutoff']:.8g}.")
del X_val, y_val
gc.collect()


Encoding features:   0%|          | 0/8 [00:00<?, ?it/s]

Choosing feature and cutoff:   0%|          | 0/4096 [00:00<?, ?it/s]

,feature,rule,activation_cutoff,precision,recall,f1
0,2126,>=,0.065555,0.413174,0.251825,0.312925
1,3060,<=,-0.058207,0.173913,0.109489,0.134378
2,1603,>=,0.051934,0.133459,0.129562,0.131481
3,3260,<=,-0.093678,0.117910,0.144161,0.129721
4,1635,>=,0.055217,0.230392,0.085766,0.125000


Selected feature 2126: predict cyclone when activation >= 0.065555006.


39

## 4. Evaluate the saved choice on 2021 storms

The test cell reads the feature ID and cutoff from disk and applies them as saved. The test F1 is your estimate for unseen storms in this sampled experiment. The proportions of positive and quiet snapshots were set in advance for the course run.

Only the selected feature is retained for the test split, so test evaluation does not need the full KAN feature cache.


In [5]:
# @title Measure precision, recall and F1 on the held-out group
selection = json.loads((OUT / 'selected_feature.json').read_text())
if selection['experiment'] != identity:
    raise ValueError('The selection belongs to a different checkpoint or labeled dataset.')
winner = selection['validation']
X_test, y_test, centers = encode_split('test', LABEL_RUN, manifest, y_by_time, model, device, selected_feature=winner['feature'])
test, by_time = evaluate_selected(X_test, y_test, centers, winner)
write_json(OUT / 'test_metrics.json',
           dict(experiment=identity, feature=winner['feature'], rule=winner['rule'],
                activation_cutoff=winner['activation_cutoff'], test=test))
write_table(OUT / 'test_by_time.csv', by_time)
fields = ['tp','fp','fn','tn','precision','recall','f1']
display(pd.DataFrame([{k: s[k] for k in fields} for s in (winner, test)],
                     index=['2020 validation (chosen here)', '2021 test (locked choice)']))
print('Feature, rule and validation result:', OUT / 'selected_feature.json')
print('Held-out result:', OUT / 'test_metrics.json')
release_features(X_test)
del X_test, y_test
gc.collect()

Encoding features:   0%|          | 0/8 [00:00<?, ?it/s]

,tp,fp,fn,tn,precision,recall,f1
2020 validation (chosen here),138,196,410,326952,0.413174,0.251825,0.312925
2021 test (locked choice),177,113,383,327023,0.610345,0.316071,0.416471


Feature, rule and validation result: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/kan_topk_k32_13ce05f84137/selected_feature.json
Held-out result: /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course/probe_runs/kan_topk_k32_13ce05f84137/test_metrics.json


86

`fitted_probes.csv` contains all slopes and intercepts. `validation_ranking.csv` shows how the candidate features ranked. `test_by_time.csv` lets you inspect hits, misses and false alarms on each 2021 map. Feature numbers belong to the checkpoint that produced them. To compare a different SAE, use the same labeled times, splits and metrics.

[MacMillan & Ouellette, single-feature probes](https://arxiv.org/html/2512.24440v1#S2.SS5) · [ClimateNetLarge labeling methods](https://doi.org/10.1038/s41597-025-05480-0)